# Homework 4 — Stock Prediction and Trading Strategy

**Hiba Lina Ben Khaddah**

This notebook develops, validates, and freezes the stock-selection research design. A final backtest notebook will evaluate the frozen strategy against the S&P 500 total return index.

## Experiment design

The workflow was first tested with the classroom database. This submitted research uses the full homework database, develops the model on 2010–2015 data, and freezes every decision before the April 2016–September 2026 final test.

In [1]:
from pathlib import Path
import sys

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    """Find the course repository from either the root or notebook folder."""
    for candidate in (start, *start.parents):
        if (candidate / "Lectures/modeling/course_data.py").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the course repository root")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
MODELING_DIR = REPO_ROOT / "Lectures/modeling"
FULL_DB = MODELING_DIR / "data/equities_top2000.duckdb"

if str(MODELING_DIR) not in sys.path:
    sys.path.insert(0, str(MODELING_DIR))
if not FULL_DB.is_file():
    raise FileNotFoundError(f"Database not found: {FULL_DB}")

print(f"Repository: {REPO_ROOT}")
print(f"Homework database: {FULL_DB.name}")

Repository: /Users/hb525/Desktop/MFE PreProgram/UCB-MFE-python-preprogram-2026-winter
Homework database: equities_top2000.duckdb


## 1. Build the full development universe

Use monthly signals from the dated top-2,000 membership. Features use only information available at the signal close, and labels match the next-session entry and 90-calendar-day holding period.

In [2]:
DEVELOPMENT_START = pd.Timestamp("2010-01-01")
DEVELOPMENT_END = pd.Timestamp("2015-12-31")
BUFFER_END = pd.Timestamp("2016-03-31")

with duckdb.connect(str(FULL_DB), read_only=True) as connection:
    # Calendar is restricted to development plus the target-completion buffer.
    benchmark_calendar = connection.execute(
        """
        SELECT date
        FROM benchmark
        WHERE date <= ?
        ORDER BY date
        """,
        [BUFFER_END],
    ).fetchdf()["date"]

    # Include securities eligible during development. Q1 2016 membership is
    # needed when a December 2015 signal enters on the next session.
    development_symbols = connection.execute(
        """
        SELECT DISTINCT symbol
        FROM universe_membership
        WHERE quarter_end >= ?
          AND quarter_start <= DATE '2016-01-01'
        ORDER BY symbol
        """,
        [DEVELOPMENT_START],
    ).fetchdf()["symbol"].tolist()

benchmark_calendar = pd.DatetimeIndex(benchmark_calendar).astype("datetime64[ns]")

development_sessions = benchmark_calendar[
    (benchmark_calendar >= DEVELOPMENT_START)
    & (benchmark_calendar <= DEVELOPMENT_END)
]

# Use the final market session of each month as the signal date.
monthly_signal_dates = (
    pd.Series(development_sessions, index=development_sessions)
    .groupby(development_sessions.to_period("M"))
    .max()
    .to_numpy()
)
monthly_signal_dates = pd.DatetimeIndex(monthly_signal_dates)

assert benchmark_calendar.max() <= BUFFER_END
assert monthly_signal_dates.min() >= DEVELOPMENT_START
assert monthly_signal_dates.max() <= DEVELOPMENT_END

display(
    pd.Series(
        {
            "development_symbols": len(development_symbols),
            "calendar_end": benchmark_calendar.max(),
            "monthly_signal_dates": len(monthly_signal_dates),
            "first_signal": monthly_signal_dates.min(),
            "last_signal": monthly_signal_dates.max(),
        },
        name="full_development_design",
    )
)

development_symbols                    2512
calendar_end            2016-03-31 00:00:00
monthly_signal_dates                     72
first_signal            2010-01-29 00:00:00
last_signal             2015-12-31 00:00:00
Name: full_development_design, dtype: object

### Construct the full development panel in batches

The full database is processed in ticker batches to limit memory use. Price features are calculated from each stock’s complete available history, but only monthly development signals are retained. Financial information and quarterly membership are joined using their historical availability dates. Missing future targets are preserved and are not used to screen securities. The lecture loader also quarantines one off-calendar observation for PRDO on September 11, 2001 rather than treating it as a valid market session.

In [3]:
from course_data import (
    FEATURES,
    FINANCIAL_FEATURES,
    PRICE_FEATURES,
    apply_membership,
    date_folds,
    financial_events,
    join_financials,
    load_data,
    load_membership,
    make_labels,
    make_pipeline,
    prediction_metrics,
    price_features,
)

BATCH_SIZE = 200
full_membership = load_membership(FULL_DB, end=BUFFER_END)
development_batches = []

for batch_start in range(0, len(development_symbols), BATCH_SIZE):
    batch_symbols = development_symbols[batch_start : batch_start + BATCH_SIZE]
    batch_prices, batch_fundamentals, batch_universe, _ = load_data(
        FULL_DB, end=BUFFER_END, symbols=batch_symbols
    )
    batch_panel = price_features(batch_prices, benchmark_calendar)
    batch_panel = batch_panel.loc[batch_panel["date"].isin(monthly_signal_dates)].copy()
    batch_panel = join_financials(
        batch_panel, financial_events(batch_fundamentals), batch_universe
    )
    batch_panel = apply_membership(
        batch_panel, full_membership, benchmark_calendar
    )
    batch_panel = make_labels(batch_panel, batch_prices, benchmark_calendar)
    development_batches.append(batch_panel)
    completed = min(batch_start + BATCH_SIZE, len(development_symbols))
    print(f"Processed {completed:,} of {len(development_symbols):,} symbols")

full_development_panel = pd.concat(
    development_batches, ignore_index=True
).sort_values(["date", "symbol"]).reset_index(drop=True)
del development_batches

display(pd.Series({
    "monthly_rows": len(full_development_panel),
    "symbols": full_development_panel["symbol"].nunique(),
    "eligible_rows": int(full_development_panel["eligible"].sum()),
    "completed_targets": int(full_development_panel["target"].notna().sum()),
    "first_signal": full_development_panel["date"].min(),
    "last_signal": full_development_panel["date"].max(),
    "latest_target_completion": full_development_panel["label_end"].max(),
}, name="full_development_panel"))

Processed 200 of 2,512 symbols


Processed 400 of 2,512 symbols


Processed 600 of 2,512 symbols


Processed 800 of 2,512 symbols


Processed 1,000 of 2,512 symbols


Processed 1,200 of 2,512 symbols


... (5 outputs left out)

Processed 2,200 of 2,512 symbols


Processed 2,400 of 2,512 symbols


Processed 2,512 of 2,512 symbols


monthly_rows                             158187
symbols                                    2512
eligible_rows                            141562
completed_targets                        155416
first_signal                2010-01-29 00:00:00
last_signal                 2015-12-31 00:00:00
latest_target_completion    2016-03-04 00:00:00
Name: full_development_panel, dtype: object

### Audit the full modeling sample

Model fitting requires historically eligible rows with completed 90-day outcomes. Rows with incomplete outcomes remain in the full panel and may later be prediction candidates, but they cannot be training observations. This separation avoids selecting stocks based on future return availability.

In [4]:
modeling_mask = (
    full_development_panel["eligible"]
    & full_development_panel["target"].notna()
)

full_modeling_data = (
    full_development_panel.loc[modeling_mask]
    .copy()
    .reset_index(drop=True)
)

eligible_without_target = (
    full_development_panel["eligible"]
    & full_development_panel["target"].isna()
)

# Verify causal timing and the development/buffer boundaries.
assert full_modeling_data["date"].between(
    DEVELOPMENT_START,
    DEVELOPMENT_END,
).all()
assert full_modeling_data["label_end"].le(BUFFER_END).all()
assert (
    full_modeling_data["available_date"].isna()
    | full_modeling_data["available_date"].le(full_modeling_data["date"])
).all()
assert full_modeling_data["universe_selection_date"].le(
    full_modeling_data["date"]
).all()
assert np.isfinite(full_modeling_data["target"]).all()
assert full_modeling_data["target"].gt(-1).all()

yearly_coverage = (
    full_modeling_data
    .assign(signal_year=full_modeling_data["date"].dt.year)
    .groupby("signal_year")
    .agg(
        rows=("symbol", "size"),
        symbols=("symbol", "nunique"),
        signal_dates=("date", "nunique"),
        mean_target=("target", "mean"),
        target_std=("target", "std"),
    )
)

full_feature_missingness = (
    full_modeling_data[FEATURES]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_fraction")
)
full_target_summary = full_modeling_data["target"].describe(
    percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]
)

display(
    pd.Series(
        {
            "training_rows": len(full_modeling_data),
            "training_symbols": full_modeling_data["symbol"].nunique(),
            "eligible_rows_without_completed_target": int(
                eligible_without_target.sum()
            ),
            "first_training_signal": full_modeling_data["date"].min(),
            "last_training_signal": full_modeling_data["date"].max(),
            "last_completed_exit": full_modeling_data["label_end"].max(),
        },
        name="full_modeling_sample",
    )
)
display(yearly_coverage.round(4))
display(full_feature_missingness.to_frame())
display(full_target_summary.to_frame("90_day_return"))

training_rows                                          139554
training_symbols                                         2459
eligible_rows_without_completed_target                   2008
first_training_signal                     2010-01-29 00:00:00
last_training_signal                      2015-11-30 00:00:00
last_completed_exit                       2016-02-29 00:00:00
Name: full_modeling_sample, dtype: object

,rows,symbols,signal_dates,mean_target,target_std
signal_year,,,,,
2010,22747,1945,12,0.1605,8.2133
2011,23540,2006,12,2.3410,354.6465
2012,23710,2063,12,4.5526,693.9029
2013,23804,2102,12,0.0796,0.3168
2014,23851,2140,12,0.0281,0.1691
2015,21902,2111,11,-0.0255,0.1910


,missing_fraction
drawdown_252,0.032590
momentum_252_21,0.029809
net_margin,0.021117
fcf_to_assets,0.004500
accruals_to_assets,0.004500
debt_to_assets,0.001877
operating_roa,0.001813
cash_to_assets,0.001813
momentum_21,0.000000
momentum_63,0.000000


,90_day_return
count,139554.000000
mean,1.208883
std,320.986439
min,-0.999992
1%,-0.446576
5%,-0.254545
50%,0.028715
95%,0.333333
99%,0.660213
max,106847.563969


### Audit extreme target returns

The full-universe target distribution contains economically implausible extremes. Before choosing a treatment, the following audit identifies their symbols, dates, and raw-versus-adjusted entry and exit prices. This cell is diagnostic only: it does not delete, cap, or modify any observation.

In [5]:
target_threshold_counts = pd.Series(
    {
        "return_above_100_percent": int(full_modeling_data["target"].gt(1).sum()),
        "return_above_500_percent": int(full_modeling_data["target"].gt(5).sum()),
        "return_above_1000_percent": int(full_modeling_data["target"].gt(10).sum()),
        "return_above_10000_percent": int(full_modeling_data["target"].gt(100).sum()),
        "loss_below_95_percent": int(full_modeling_data["target"].lt(-0.95).sum()),
    },
    name="target_outlier_counts",
)

largest_targets = (
    full_modeling_data.nlargest(15, "target")
    [[
        "symbol",
        "sector",
        "date",
        "entry_date",
        "entry_price",
        "scheduled_exit",
        "label_end",
        "exit_price",
        "target",
        "universe_rank",
    ]]
    .copy()
)
largest_targets["target_percent"] = 100 * largest_targets["target"]

# Retrieve raw and adjusted prices for the audited entry and exit dates.
entry_dates = largest_targets[["date", "symbol", "entry_date"]].rename(
    columns={"date": "signal_date", "entry_date": "price_date"}
)
entry_dates["price_role"] = "entry"
exit_dates = largest_targets[["date", "symbol", "label_end"]].rename(
    columns={"date": "signal_date", "label_end": "price_date"}
)
exit_dates["price_role"] = "exit"
extreme_dates = pd.concat([entry_dates, exit_dates], ignore_index=True)

with duckdb.connect(str(FULL_DB), read_only=True) as connection:
    connection.register("extreme_dates", extreme_dates)
    extreme_price_audit = connection.execute(
        """
        SELECT
            e.signal_date,
            e.symbol,
            e.price_role,
            p.date AS price_date,
            p.close AS raw_close,
            p.adj_close,
            p.close / p.adj_close AS raw_to_adjusted_ratio
        FROM extreme_dates AS e
        LEFT JOIN prices AS p
          ON e.symbol = p.symbol
         AND e.price_date = p.date
        ORDER BY e.signal_date, e.symbol, e.price_role
        """
    ).fetchdf()

display(target_threshold_counts)
display(largest_targets.round(4))
display(extreme_price_audit.round(6))

return_above_100_percent      476
return_above_500_percent       24
return_above_1000_percent      17
return_above_10000_percent      5
loss_below_95_percent           5
Name: target_outlier_counts, dtype: int64

,symbol,sector,date,entry_date,entry_price,scheduled_exit,label_end,exit_price,target,universe_rank,target_percent
52214,AAM,Financial Services,2012-04-30,2012-05-01,1.915000e-01,2012-07-30,2012-07-30,2.046150e+04,106847.5640,1.0,1.068476e+07
28573,AAM,Financial Services,2011-04-29,2011-05-02,4.135000e-01,2011-07-31,2011-08-01,2.250000e+04,54412.5429,1.0,5.441254e+06
1748,VBIV,Healthcare,2010-01-29,2010-02-01,1.145400e+00,2010-05-02,2010-05-03,9.360000e+02,816.2140,1888.0,8.162140e+04
5484,VBIV,Healthcare,2010-03-31,2010-04-01,1.106400e+00,2010-06-30,2010-06-30,7.320000e+02,660.6069,1901.0,6.606069e+04
3611,VBIV,Healthcare,2010-02-26,2010-03-01,1.223300e+00,2010-05-30,2010-06-01,8.040000e+02,656.2128,1888.0,6.562128e+04
88887,LCI,Healthcare,2013-10-31,2013-11-01,2.430000e+01,2014-01-30,2014-01-30,5.760000e+02,22.7037,816.0,2.270370e+03
86901,LCI,Healthcare,2013-09-30,2013-10-01,2.235000e+01,2013-12-30,2013-12-30,5.252800e+02,22.5025,816.0,2.250246e+03
90875,LCI,Healthcare,2013-11-29,2013-12-02,3.210000e+01,2014-03-02,2014-03-03,6.932800e+02,20.5975,816.0,2.059751e+03
60751,FBC,Financial Services,2012-08-31,2012-09-04,9.475000e-01,2012-12-03,2012-12-03,1.855000e+01,18.5778,1896.0,1.857784e+03
63294,MULN,Consumer Cyclical,2012-09-28,2012-10-01,4.050000e+07,2012-12-30,2012-12-31,7.852500e+08,18.3889,1973.0,1.838889e+03


,signal_date,symbol,price_role,price_date,raw_close,adj_close,raw_to_adjusted_ratio
0,2010-01-29,VBIV,entry,2010-02-01,1.145355e+00,1.145355e+00,1.00
1,2010-01-29,VBIV,exit,2010-05-03,9.360000e+02,9.360000e+02,1.00
2,2010-02-26,VBIV,entry,2010-03-01,1.223348e+00,1.223348e+00,1.00
3,2010-02-26,VBIV,exit,2010-06-01,8.040000e+02,8.040000e+02,1.00
4,2010-03-31,VBIV,entry,2010-04-01,1.106397e+00,1.106397e+00,1.00
5,2010-03-31,VBIV,exit,2010-06-30,7.320000e+02,7.320000e+02,1.00
6,2010-09-30,PALT,entry,2010-10-01,5.250000e+00,5.250000e+00,1.00
7,2010-09-30,PALT,exit,2010-12-30,7.000000e+01,7.000000e+01,1.00
8,2010-11-30,PALT,entry,2010-12-01,7.000000e+00,7.000000e+00,1.00
9,2010-11-30,PALT,exit,2011-03-01,1.057000e+02,1.057000e+02,1.00


### Flag labels crossing price discontinuities

Following the lecture principle that suspicious source values should be investigated rather than silently repaired, original prices and targets are preserved. A one-session adjusted-price ratio above 10 or below 0.1 is flagged as an apparent corporate-action, unit, or ticker-history discontinuity. A completed 90-day label crossing such an event is excluded from model fitting and prediction evaluation, while remaining available in the audit data. This is a documented extension for the full homework database; the lecture pipeline itself does not clip targets.

In [6]:
with duckdb.connect(str(FULL_DB), read_only=True) as connection:
    price_discontinuities = connection.execute(
        """
        WITH relevant_symbols AS (
            SELECT DISTINCT symbol
            FROM universe_membership
            WHERE quarter_end >= ?
              AND quarter_start <= DATE '2016-01-01'
        ),
        ordered_prices AS (
            SELECT
                p.symbol,
                p.date,
                p.adj_close,
                LAG(p.adj_close) OVER (
                    PARTITION BY p.symbol ORDER BY p.date
                ) AS previous_adj_close
            FROM prices AS p
            INNER JOIN relevant_symbols AS r USING (symbol)
            WHERE p.date <= ?
              AND p.adj_close > 0
        )
        SELECT
            symbol,
            date,
            previous_adj_close,
            adj_close,
            adj_close / previous_adj_close AS one_session_price_ratio
        FROM ordered_prices
        WHERE previous_adj_close > 0
          AND (
              adj_close / previous_adj_close > 10
              OR adj_close / previous_adj_close < 0.1
          )
        ORDER BY date, symbol
        """,
        [DEVELOPMENT_START, BUFFER_END],
    ).fetchdf()

price_discontinuities["date"] = pd.to_datetime(
    price_discontinuities["date"]
).astype("datetime64[ns]")

crosses_discontinuity = pd.Series(
    False,
    index=full_development_panel.index,
)
for event in price_discontinuities.itertuples(index=False):
    event_date = pd.Timestamp(event.date)
    crosses_discontinuity |= (
        full_development_panel["symbol"].eq(event.symbol)
        & full_development_panel["entry_date"].lt(event_date)
        & full_development_panel["label_end"].ge(event_date)
    )

full_development_panel["crosses_price_discontinuity"] = (
    crosses_discontinuity
)
reliable_modeling_mask = modeling_mask & ~crosses_discontinuity
reliable_modeling_data = (
    full_development_panel.loc[reliable_modeling_mask]
    .copy()
    .reset_index(drop=True)
)
unreliable_labels = full_development_panel.loc[
    modeling_mask & crosses_discontinuity,
    [
        "symbol", "date", "entry_date", "label_end",
        "entry_price", "exit_price", "target",
    ],
].copy()

assert not reliable_modeling_data["crosses_price_discontinuity"].any()

discontinuity_summary = pd.Series(
    {
        "one_session_discontinuities": len(price_discontinuities),
        "symbols_with_discontinuities": price_discontinuities["symbol"].nunique(),
        "completed_labels_flagged": int(crosses_discontinuity[modeling_mask].sum()),
        "symbols_with_flagged_labels": unreliable_labels["symbol"].nunique(),
        "reliable_modeling_rows": len(reliable_modeling_data),
        "reliable_target_median": reliable_modeling_data["target"].median(),
        "reliable_target_maximum": reliable_modeling_data["target"].max(),
    },
    name="price_discontinuity_audit",
)

display(discontinuity_summary)
display(unreliable_labels.sort_values("target", ascending=False).head(20).round(4))

one_session_discontinuities        171.000000
symbols_with_discontinuities        37.000000
completed_labels_flagged            40.000000
symbols_with_flagged_labels          9.000000
reliable_modeling_rows          139514.000000
reliable_target_median               0.028711
reliable_target_maximum             14.100000
Name: price_discontinuity_audit, dtype: float64

,symbol,date,entry_date,label_end,entry_price,exit_price,target
54567,AAM,2012-04-30,2012-05-01,2012-07-30,1.915000e-01,2.046150e+04,106847.5640
29837,AAM,2011-04-29,2011-05-02,2011-08-01,4.135000e-01,2.250000e+04,54412.5429
1825,VBIV,2010-01-29,2010-02-01,2010-05-03,1.145400e+00,9.360000e+02,816.2140
5734,VBIV,2010-03-31,2010-04-01,2010-06-30,1.106400e+00,7.320000e+02,660.6069
3777,VBIV,2010-02-26,2010-03-01,2010-06-01,1.223300e+00,8.040000e+02,656.2128
94726,LCI,2013-10-31,2013-11-01,2014-01-30,2.430000e+01,5.760000e+02,22.7037
92474,LCI,2013-09-30,2013-10-01,2013-12-30,2.235000e+01,5.252800e+02,22.5025
96993,LCI,2013-11-29,2013-12-02,2014-03-03,3.210000e+01,6.932800e+02,20.5975
63703,FBC,2012-08-31,2012-09-04,2012-12-03,9.475000e-01,1.855000e+01,18.5778
66448,MULN,2012-09-28,2012-10-01,2012-12-31,4.050000e+07,7.852500e+08,18.3889


## 2. Chronological validation

Validation years are 2013, 2014, and 2015. Every training fold uses only 90-day outcomes completed before its validation year.

In [7]:
reliable_target_by_year = (
    reliable_modeling_data
    .assign(signal_year=reliable_modeling_data["date"].dt.year)
    .groupby("signal_year")["target"]
    .agg(
        rows="size",
        mean="mean",
        median="median",
        standard_deviation="std",
        maximum="max",
    )
)

FULL_VALIDATION_YEARS = (2013, 2014, 2015)
full_folds = date_folds(
    reliable_modeling_data,
    validation_years=FULL_VALIDATION_YEARS,
    train_start=DEVELOPMENT_START,
)

full_fold_audit_rows = []
for validation_year, (train_index, validation_index) in zip(
    FULL_VALIDATION_YEARS,
    full_folds,
):
    training_fold = reliable_modeling_data.iloc[train_index]
    validation_fold = reliable_modeling_data.iloc[validation_index]
    first_validation_signal = validation_fold["date"].min()

    assert training_fold["date"].max() < first_validation_signal
    assert training_fold["label_end"].max() < first_validation_signal
    assert not set(training_fold["date"]).intersection(validation_fold["date"])
    assert not training_fold["crosses_price_discontinuity"].any()
    assert not validation_fold["crosses_price_discontinuity"].any()

    full_fold_audit_rows.append(
        {
            "validation_year": validation_year,
            "training_rows": len(training_fold),
            "training_signal_dates": training_fold["date"].nunique(),
            "validation_rows": len(validation_fold),
            "validation_signal_dates": validation_fold["date"].nunique(),
            "last_training_signal": training_fold["date"].max(),
            "last_completed_training_exit": training_fold["label_end"].max(),
            "first_validation_signal": first_validation_signal,
            "last_validation_signal": validation_fold["date"].max(),
        }
    )

full_fold_audit = pd.DataFrame(full_fold_audit_rows)

display(reliable_target_by_year.round(4))
display(full_fold_audit)

,rows,mean,median,standard_deviation,maximum
signal_year,,,,,
2010,22735,0.0662,0.0471,0.2649,14.1000
2011,23529,0.0281,0.0194,0.2379,5.9091
2012,23702,0.0434,0.0363,0.2016,4.2308
2013,23801,0.0768,0.0558,0.2000,6.6036
2014,23845,0.0282,0.0264,0.1689,3.5435
2015,21902,-0.0255,-0.0201,0.1910,8.5116


,validation_year,training_rows,training_signal_dates,validation_rows,validation_signal_dates,last_training_signal,last_completed_training_exit,first_validation_signal,last_validation_signal
0,2013,66015,34,23801,12,2012-10-31,2013-01-30,2013-01-31,2013-12-31
1,2014,89808,46,23845,12,2013-10-31,2014-01-30,2014-01-31,2014-12-31
2,2015,111648,57,21902,11,2014-09-30,2014-12-30,2015-01-30,2015-11-30


## 3. Compare model candidates

The Lecture 7 models use identical features, chronological folds, and training-fitted preprocessing. Mean daily Spearman rank IC is the main metric because the strategy buys the highest-ranked stocks.

In [8]:
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge


candidates = {
    "training mean": make_pipeline(DummyRegressor(strategy="mean")),
    "OLS": make_pipeline(LinearRegression()),
    "Ridge": make_pipeline(Ridge(alpha=100)),
    "Lasso": make_pipeline(Lasso(alpha=0.001, max_iter=10_000)),
    "boosted trees": make_pipeline(
        HistGradientBoostingRegressor(
            max_iter=60,
            max_leaf_nodes=7,
            min_samples_leaf=100,
            l2_regularization=10,
            early_stopping=False,
            random_state=42,
        )
    ),
}

full_X = reliable_modeling_data[FEATURES]
full_y = reliable_modeling_data["target"]
full_model_results = []

for model_name, estimator in candidates.items():
    for fold_number, (train_index, validation_index) in enumerate(full_folds, start=1):
        fitted_model = clone(estimator).fit(
            full_X.iloc[train_index], full_y.iloc[train_index]
        )
        predictions = fitted_model.predict(full_X.iloc[validation_index])
        validation_frame = reliable_modeling_data.iloc[validation_index]
        metrics = prediction_metrics(validation_frame, predictions)
        full_model_results.append({
            "model": model_name,
            "fold": fold_number,
            "validation_year": FULL_VALIDATION_YEARS[fold_number - 1],
            **metrics.to_dict(),
        })

full_model_results = pd.DataFrame(full_model_results)
full_model_summary = (
    full_model_results.groupby("model").agg(
        average_RMSE=("RMSE", "mean"),
        average_MAE=("MAE", "mean"),
        average_R2=("R2", "mean"),
        average_rank_IC=("mean_daily_rank_IC", "mean"),
        rank_IC_std=("mean_daily_rank_IC", "std"),
        minimum_rank_IC=("mean_daily_rank_IC", "min"),
    ).sort_values("average_rank_IC", ascending=False)
)

display(full_model_results.round(4))
display(full_model_summary.round(4))

,model,fold,validation_year,RMSE,MAE,R2,mean_daily_rank_IC,IC_dates
0,training mean,1,2013,0.2031,0.1208,-0.0313,NaN,0.0
1,training mean,2,2014,0.1708,0.1153,-0.0220,NaN,0.0
2,training mean,3,2015,0.2050,0.1433,-0.1519,NaN,0.0
3,OLS,1,2013,0.2088,0.1253,-0.0894,-0.0085,12.0
4,OLS,2,2014,0.1712,0.1148,-0.0279,-0.0105,12.0
5,OLS,3,2015,0.2068,0.1443,-0.1727,-0.0699,11.0
6,Ridge,1,2013,0.2088,0.1253,-0.0892,-0.0084,12.0
7,Ridge,2,2014,0.1712,0.1148,-0.0278,-0.0106,12.0
8,Ridge,3,2015,0.2068,0.1443,-0.1727,-0.0703,11.0
9,Lasso,1,2013,0.2077,0.1242,-0.0780,0.0052,12.0


,average_RMSE,average_MAE,average_R2,average_rank_IC,rank_IC_std,minimum_rank_IC
model,,,,,,
boosted trees,0.1955,0.1278,-0.0955,-0.0198,0.0348,-0.0599
OLS,0.1956,0.1281,-0.0967,-0.0297,0.0349,-0.0699
Ridge,0.1956,0.1281,-0.0965,-0.0298,0.0351,-0.0703
Lasso,0.1951,0.1276,-0.0910,-0.0382,0.0515,-0.0951
training mean,0.1930,0.1264,-0.0684,NaN,NaN,NaN


## 4. Final development comparison

Raw-return models produced negative ranking IC because extreme returns strongly influenced squared-error fitting. We therefore predict each stock’s within-date return rank and compare rank-target boosted trees with a signed composite and the Lecture 8 momentum baseline. A relative-return target was also tested but did not improve ranking performance, so that experiment is not shown.

In [9]:
diagnostic_features = PRICE_FEATURES + FINANCIAL_FEATURES + ["filing_age_days"]

signal_calendar_positions = pd.Series(
    benchmark_calendar.get_indexer(full_development_panel["date"]),
    index=full_development_panel.index,
)
known_discontinuity_previous_252 = pd.Series(
    False,
    index=full_development_panel.index,
)

for event in price_discontinuities.itertuples(index=False):
    event_position = benchmark_calendar.get_indexer([pd.Timestamp(event.date)])[0]
    if event_position < 0:
        continue
    sessions_since_event = signal_calendar_positions - event_position
    known_discontinuity_previous_252 |= (
        full_development_panel["symbol"].eq(event.symbol)
        & sessions_since_event.between(0, 252)
    )

full_development_panel["known_discontinuity_previous_252"] = (
    known_discontinuity_previous_252
)

first_2013_signal = reliable_modeling_data.loc[
    reliable_modeling_data["date"].dt.year.eq(2013),
    "date",
].min()
purged_selection_mask = (
    full_development_panel["eligible"]
    & full_development_panel["target"].notna()
    & full_development_panel["label_end"].lt(first_2013_signal)
    & ~full_development_panel["crosses_price_discontinuity"]
    & ~full_development_panel["known_discontinuity_previous_252"]
)
purged_selection_data = full_development_panel.loc[purged_selection_mask].copy()

prevalidation_ic_records = []
for feature in diagnostic_features:
    for signal_date, group in purged_selection_data.groupby("date"):
        observed = group[[feature, "target"]].dropna()
        if len(observed) >= 20 and observed[feature].nunique() > 1:
            prevalidation_ic_records.append(
                {
                    "feature": feature,
                    "date": signal_date,
                    "year": signal_date.year,
                    "rank_IC": observed[feature].corr(
                        observed["target"],
                        method="spearman",
                    ),
                }
            )

prevalidation_ic = pd.DataFrame(prevalidation_ic_records)
prevalidation_yearly_ic = (
    prevalidation_ic
    .pivot_table(index="feature", columns="year", values="rank_IC", aggfunc="mean")
    .reindex(columns=[2010, 2011, 2012])
)
prevalidation_feature_summary = prevalidation_yearly_ic.copy()
prevalidation_feature_summary["mean_IC"] = (
    prevalidation_ic.groupby("feature")["rank_IC"].mean()
)
prevalidation_feature_summary["consistent_sign"] = (
    prevalidation_yearly_ic.gt(0).all(axis=1)
    | prevalidation_yearly_ic.lt(0).all(axis=1)
)
prevalidation_feature_summary["selected"] = (
    prevalidation_feature_summary["consistent_sign"]
    & prevalidation_feature_summary["mean_IC"].abs().ge(0.02)
)
selected_composite_features = prevalidation_feature_summary.index[
    prevalidation_feature_summary["selected"]
].tolist()
selected_feature_signs = np.sign(
    prevalidation_feature_summary.loc[selected_composite_features, "mean_IC"]
)

assert purged_selection_data["label_end"].max() < first_2013_signal
assert selected_composite_features
display(
    pd.Series(
        {
            "first_validation_signal": first_2013_signal,
            "purged_feature_selection_rows": len(purged_selection_data),
            "monthly_rows_flagged_by_causal_screen": int(
                known_discontinuity_previous_252.sum()
            ),
            "selected_feature_count": len(selected_composite_features),
        },
        name="final_development_setup",
    )
)
display(prevalidation_feature_summary.round(4))
print("Selected composite features:", selected_composite_features)
print("Selected feature signs:", selected_feature_signs.to_dict())

first_validation_signal                  2013-01-31 00:00:00
purged_feature_selection_rows                          65949
monthly_rows_flagged_by_causal_screen                    280
selected_feature_count                                     7
Name: final_development_setup, dtype: object

year,2010,2011,2012,mean_IC,consistent_sign,selected
feature,,,,,,
accruals_to_assets,0.0051,-0.0075,0.0334,0.0090,False,False
cash_to_assets,-0.0112,-0.0474,-0.0856,-0.0458,True,True
debt_to_assets,0.0207,0.0375,0.0867,0.0461,True,True
drawdown_252,-0.0030,0.0363,0.0690,0.0320,False,False
fcf_to_assets,0.0373,0.0945,0.0201,0.0524,True,True
filing_age_days,0.0173,0.0158,0.0190,0.0173,True,False
momentum_21,-0.0187,-0.0100,0.0067,-0.0082,False,False
momentum_252_21,0.0134,-0.0073,0.0830,0.0265,False,False
momentum_63,-0.0331,-0.0155,-0.0151,-0.0216,True,True


Selected composite features: ['cash_to_assets', 'debt_to_assets', 'fcf_to_assets', 'momentum_63', 'net_margin', 'operating_roa', 'volatility_63']
Selected feature signs: {'cash_to_assets': -1.0, 'debt_to_assets': 1.0, 'fcf_to_assets': 1.0, 'momentum_63': -1.0, 'net_margin': 1.0, 'operating_roa': 1.0, 'volatility_63': -1.0}


In [10]:
comparison_mask = (
    full_development_panel["eligible"]
    & full_development_panel["target"].notna()
    & ~full_development_panel["crosses_price_discontinuity"]
    & ~full_development_panel["known_discontinuity_previous_252"]
)
comparison_data = (
    full_development_panel.loc[comparison_mask]
    .copy()
    .reset_index(drop=True)
)
comparison_data["rank_target"] = (
    comparison_data.groupby("date")["target"].rank(pct=True) - 0.5
)
comparison_folds = date_folds(
    comparison_data,
    validation_years=FULL_VALIDATION_YEARS,
    train_start=DEVELOPMENT_START,
)
rank_tree_estimator = candidates["boosted trees"]
development_score_rows = []

for fold_number, (train_index, validation_index) in enumerate(
    comparison_folds,
    start=1,
):
    validation_rows = comparison_data.iloc[validation_index]
    validation_dates = validation_rows["date"].unique()
    scoring_candidates = full_development_panel.loc[
        full_development_panel["date"].isin(validation_dates)
        & full_development_panel["eligible"]
        & ~full_development_panel["known_discontinuity_previous_252"]
    ].copy()

    rank_tree = clone(rank_tree_estimator).fit(
        comparison_data.iloc[train_index][FEATURES],
        comparison_data.iloc[train_index]["rank_target"],
    )
    scoring_candidates["rank-target boosted trees"] = rank_tree.predict(
        scoring_candidates[FEATURES]
    )

    signed_ranks = []
    for feature in selected_composite_features:
        higher_is_better = selected_feature_signs[feature] > 0
        signed_rank = scoring_candidates.groupby("date")[feature].rank(
            pct=True,
            ascending=higher_is_better,
        )
        signed_ranks.append(signed_rank.fillna(0.5))
    scoring_candidates["signed composite"] = pd.concat(
        signed_ranks,
        axis=1,
    ).mean(axis=1)
    scoring_candidates["63-session momentum"] = scoring_candidates["momentum_63"]

    realized = validation_rows[["date", "symbol", "target"]]
    for strategy_name in (
        "rank-target boosted trees",
        "signed composite",
        "63-session momentum",
    ):
        scored = realized.merge(
            scoring_candidates[["date", "symbol", strategy_name]],
            on=["date", "symbol"],
            how="inner",
            validate="one_to_one",
        ).rename(columns={strategy_name: "score"})
        scored["strategy"] = strategy_name
        scored["fold"] = fold_number
        scored["validation_year"] = FULL_VALIDATION_YEARS[fold_number - 1]
        development_score_rows.append(scored)

development_scores = pd.concat(development_score_rows, ignore_index=True)
development_monthly_ic = (
    development_scores
    .groupby(["strategy", "validation_year", "date"])
    .apply(
        lambda group: group["score"].corr(group["target"], method="spearman"),
        include_groups=False,
    )
    .rename("rank_IC")
    .reset_index()
)
development_ic_by_year = development_monthly_ic.pivot_table(
    index="strategy",
    columns="validation_year",
    values="rank_IC",
    aggfunc="mean",
)
development_comparison_summary = development_monthly_ic.groupby("strategy").agg(
    mean_rank_IC=("rank_IC", "mean"),
    IC_standard_deviation=("rank_IC", "std"),
    positive_month_fraction=("rank_IC", lambda values: values.gt(0).mean()),
    months=("rank_IC", "count"),
)
development_comparison_summary["IC_t_statistic"] = (
    development_comparison_summary["mean_rank_IC"]
    / (
        development_comparison_summary["IC_standard_deviation"]
        / np.sqrt(development_comparison_summary["months"])
    )
)
development_comparison_summary = (
    development_comparison_summary
    .join(development_ic_by_year.add_prefix("IC_"))
    .sort_values("mean_rank_IC", ascending=False)
)

display(development_comparison_summary.round(4))

,mean_rank_IC,IC_standard_deviation,positive_month_fraction,months,IC_t_statistic,IC_2013,IC_2014,IC_2015
strategy,,,,,,,,
signed composite,0.0474,0.1450,0.5714,35,1.9340,-0.0697,0.1087,0.1083
rank-target boosted trees,0.0356,0.1010,0.6571,35,2.0859,-0.0445,0.0673,0.0884
63-session momentum,0.0270,0.0849,0.6857,35,1.8781,0.0180,0.0257,0.0381


## Frozen research design

The following choices were fixed before examining the final test:

- **Model:** histogram gradient-boosting regressor using the Lecture 7 pipeline and all lecture features.
- **Target:** each stock’s within-date percentile rank of its 90-day return.
- **Training:** reliable completed outcomes with label_end before March 31, 2016; fit once.
- **Screen:** dated membership, enough feature history, and no 10× or 0.1× price transition in the previous 252 sessions.
- **Trading:** last monthly signal, next-session entry, top 20 equal-weighted stocks, and min(NAV / 3, cash) per cohort.
- **Holding period:** 90 calendar days, with each purchase tracked as a separate lot.
- **Costs:** 5 bps per side; cash earns no interest.
- **Missing prices:** no invented fills; delayed and unresolved exits are reported, and stale holdings remain at their last mark.
- **Test:** April 2016 through September 2026.

Boosted trees were selected because they followed the Lecture 7 pipeline and were more stable across validation months. The slightly higher composite IC was within the large month-to-month noise.